# 02 — Silver: Data Quality

**Tickets:** I-06, I-07  
**Purpose:** Run data quality assertions on the Silver table; document assumptions and findings.

---

## Setup

In [ ]:
from src.constants import (
    MAX_FARE_AMOUNT,
    MAX_TIP_AMOUNT,
    MAX_TOTAL_AMOUNT,
    MAX_TRIP_DISTANCE,
    SILVER_TABLE,
    VALID_EXTRA_VALUES,
    VALID_RATE_CODES,
)
from src.validators import (
    check_accepted_values,
    check_column_exists,
    check_max,
    check_no_nulls,
    check_non_negative,
    check_not_empty,
    check_positive,
)

print("Setup complete")

## Read Silver table

In [ ]:
silver_df = spark.read.table(SILVER_TABLE)
print(f"Silver table: {SILVER_TABLE}")
print(f"Row count: {silver_df.count():,}")
silver_df.printSchema()

## I-06 — Data quality checks

These assertions validate that I-03 (structural cleaning) and I-04 (outlier removal) have been applied correctly. Every check must pass before Gold/ML can consume the Silver table.

**Check categories:**
1. **Structural** — table not empty, required columns exist, no NULLs in key fields
2. **Business rules** — values within expected ranges after I-04 cleaning
3. **Referential integrity** — categorical codes within valid sets

In [ ]:
print("=" * 60)
print("1. STRUCTURAL CHECKS")
print("=" * 60)

# Table is not empty
check_not_empty(silver_df, "Silver")

# Required columns exist
for col in [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
    "passenger_count",
    "pickup_zone",
    "dropoff_zone",
]:
    check_column_exists(silver_df, col)

# No NULLs in key fields (guaranteed by I-03 drop_corrupt_rows)
for col in [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
]:
    check_no_nulls(silver_df, col)

In [ ]:
print("=" * 60)
print("2. BUSINESS RULE CHECKS (I-04 guarantees)")
print("=" * 60)

# trip_distance > 0 and <= MAX (I-04a, I-04d)
check_positive(silver_df, "trip_distance")
check_max(silver_df, "trip_distance", MAX_TRIP_DISTANCE)

# fare_amount > 0 and <= MAX (I-04b, I-04d)
check_positive(silver_df, "fare_amount")
check_max(silver_df, "fare_amount", MAX_FARE_AMOUNT)

# total_amount >= 0 and <= MAX (I-04b, I-04d)
check_non_negative(silver_df, "total_amount")
check_max(silver_df, "total_amount", MAX_TOTAL_AMOUNT)

# passenger_count > 0 (I-04c)
check_positive(silver_df, "passenger_count")

# tip_amount >= 0 and <= MAX (I-04f)
check_non_negative(silver_df, "tip_amount")
check_max(silver_df, "tip_amount", MAX_TIP_AMOUNT)

# extra surcharge: only valid values or NULL (I-04f)
check_accepted_values(silver_df, "extra", list(VALID_EXTRA_VALUES))

In [ ]:
print("=" * 60)
print("3. REFERENTIAL INTEGRITY CHECKS")
print("=" * 60)

# payment_type in {1, 2, 3, 4, 5, 6}
check_accepted_values(silver_df, "payment_type", [1, 2, 3, 4, 5, 6])

# rate_code_id in valid set or NULL (code 99 mapped to NULL in I-03)
check_accepted_values(silver_df, "rate_code_id", list(VALID_RATE_CODES))

# vendor_id in {1, 2}
check_accepted_values(silver_df, "vendor_id", [1, 2])

print()
print("=" * 60)
print("ALL DATA QUALITY CHECKS PASSED")
print("=" * 60)

## I-07 — Assumptions & Data Quality Log

### Cleaning actions applied (I-03 + I-04)

| # | Column(s) | Issue (from EDA Finding) | Action taken | Rows affected |
|---|-----------|--------------------------|-------------|---------------|
| 1 | `rate_code_id`, `_rescued_data` | 73% NULL due to 2015/2016 schema mismatch | COALESCE from `_rescued_data` JSON; code 99 -> NULL | 68,999,718 recovered |
| 2 | all | 772 exact duplicate rows | Deduplicated | 772 |
| 3 | `trip_distance` | 0.60% with distance = 0 | Dropped | ~564,480 |
| 4 | `fare_amount` | 0.07% with fare <= 0 | Dropped | ~62,088 |
| 5 | `total_amount` | 0.04% with total < 0 (voided/disputed) | Dropped | ~34,314 |
| 6 | `passenger_count` | 0.02% with count = 0 | Dropped | ~16,428 |
| 7 | `trip_distance`, `fare_amount`, `total_amount` | Extreme outliers (>100mi, >$500, >$1000) | Dropped | ~1,500 |
| 8 | `tip_amount` | 840 negative tips; tips up to $3.95M | Negative -> 0; capped at $200 | ~2,200 |
| 9 | `extra` | Non-standard surcharge values ($4.50, negatives) | NULLed out; only {0, 0.5, 1.0} kept | ~168,494 |
| 10 | GPS lat/lon | 1.55M rows with (0,0); ~77K outside NYC bbox | NULLed out (not dropped -- row otherwise valid) | ~1,622,050 |
| 11 | Duration | 946 negative, 101K zero, 330 >24h | Dropped negative/zero/over-24h | ~102,760 |
| 12 | `tip_amount`, `payment_type` | 1,150 rows with tip > 0 on non-card payment | **No action** -- kept; tip models should filter to `payment_type = 1` | 0 |

### Known nullable columns in Silver (by design)

| Column | Why nullable | Impact |
|--------|-------------|--------|
| `rate_code_id` | Code 99 -> NULL (1,670 rows) | Gold/ML should treat as "Unknown" |
| `pickup_latitude/longitude` | (0,0) and out-of-bbox -> NULL | `pickup_zone` also NULL; Gold demand heatmaps exclude these |
| `dropoff_latitude/longitude` | Same as pickup | `dropoff_zone` also NULL |
| `extra` | Non-standard values -> NULL | Gold revenue calcs should COALESCE to 0 |
| `store_and_fwd_flag` | Nullable in source | Low-importance field |

### Assumptions

- Bronze table is append-only (no transforms applied by I-01).
- Tip amounts for non-credit-card payments are not captured in `tip_amount` (cash tips excluded by design).
- `rate_code_id` values outside 1-6 and `payment_type` values outside 1-6 are data entry errors.
- Rows where `total_amount < 0` are voided/disputed trips and are excluded from analytics.
- `RatecodeID = 99` in rescued data is a data entry error, mapped to NULL in Silver.
- GPS coordinates of (0, 0) represent missing data. NYC bbox [-74.3, -73.7] x [40.4, 40.95] is a conservative filter.
- Trip durations under 1 second are meter errors or cancellations. Durations over 24 hours are data errors.
- `extra` should only be $0.00, $0.50 (rush hour), or $1.00 (overnight).
- Tip prediction models (BQ-4) must train on `payment_type = 1` only since cash tips are unobservable.
- Data covers Jan 2015 + Jan-Mar 2016 only (4 months with 9-month gap) -- seasonal claims are limited.